# Entity Normalization — TFT v2 Prep Task 2

Builds a clean three-table entity normalization set in `wti_thesis.db`:

| Table | Contents |
|---|---|
| `raw_entity_counts` | Full distribution of raw LLM entity strings with occurrence counts |
| `canonical_entities` | (future) Lookup: raw string → canonical name |
| `article_entities` | Long-format: article_id × canonical_entity pairs |

**Pipeline:**
1. Materialize `raw_entity_counts` from `llm_features.entities`
2. Triage: review distribution, update `ENTITY_CANONICAL` / `CANONICAL_ENTITIES` in `llm_features.py`
3. Reload module, apply `canonicalize_entities()`, verify coverage
4. Write to `article_entities`, run final verification

**Run order:** after `06_llm_features.ipynb`. Re-run whenever `llm_features` is updated or `ENTITY_CANONICAL` is revised.

**Does NOT modify** `llm_features` — that table stays as raw LLM output.

## Cell 1 — Setup and load

In [12]:
import sys, json, sqlite3
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('../03_src').resolve()))
from nlp.llm_features import ENTITY_CANONICAL, CANONICAL_ENTITIES, canonicalize_entities

DB_PATH = Path('../01_data/wti_thesis.db')

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql(
    "SELECT article_id, entities FROM llm_features WHERE usable=1 AND entities IS NOT NULL",
    conn
)
conn.close()

print(f'ENTITY_CANONICAL variants:  {len(ENTITY_CANONICAL):,}')
print(f'CANONICAL_ENTITIES:         {len(CANONICAL_ENTITIES):,}')
print(f'Usable articles with entities: {len(df):,}')
print()
print('Sample raw entities (first 5 rows):')
print(df.head())

ENTITY_CANONICAL variants:  71
CANONICAL_ENTITIES:         52
Usable articles with entities: 11,675

Sample raw entities (first 5 rows):
   article_id                                           entities
0           2  ["Russia", "Ukraine", "Israel", "Gaza", "OPEC ...
1           3  ["Opec", "Opec+", "US", "Vijay Valecha", "Cent...
2           4                                           ["OPEC"]
3           5  ["Brent crude", "U.S. West Texas Intermediate ...
4           8  ["Nigerian National Petroleum Company Limited ...


## Cell 2 — Materialize raw_entity_counts

Extract every distinct raw entity string from `llm_features.entities` (usable=1),
count occurrences across all articles, and write to `raw_entity_counts` in the DB.

Full distribution with no LIMIT — used to drive the triage step.

In [22]:
from datetime import datetime, timezone

conn = sqlite3.connect(DB_PATH)

conn.execute("DROP TABLE IF EXISTS raw_entity_counts")
conn.execute("""
    CREATE TABLE raw_entity_counts (
        raw_entity   TEXT    PRIMARY KEY,
        count        INTEGER NOT NULL,
        last_updated TEXT    NOT NULL
    )
""")

ts = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
conn.execute("""
    INSERT INTO raw_entity_counts (raw_entity, count, last_updated)
    SELECT TRIM(je.value), COUNT(*), ?
    FROM llm_features, json_each(llm_features.entities) AS je
    WHERE usable = 1 AND entities IS NOT NULL
    GROUP BY TRIM(je.value)
""", (ts,))
conn.commit()

df_raw = pd.read_sql(
    "SELECT raw_entity, count FROM raw_entity_counts ORDER BY count DESC",
    conn
)
conn.close()

print(f'Distinct raw entity strings: {len(df_raw):,}')
print(f'Total entity mentions:       {df_raw["count"].sum():,}')
print(f'Entities with count >= 100:  {(df_raw["count"] >= 100).sum()}')
print(f'Entities with count  50–99:  {((df_raw["count"] >= 50) & (df_raw["count"] < 100)).sum()}')
print(f'Entities with count  10–49:  {((df_raw["count"] >= 10) & (df_raw["count"] < 50)).sum()}')
print(f'Entities with count   1–9:   {(df_raw["count"] < 10).sum()}')
print(f'\nTable raw_entity_counts written to {DB_PATH.name}  (last_updated: {ts})')

Distinct raw entity strings: 5,807
Total entity mentions:       44,774
Entities with count >= 100:  61
Entities with count  50–99:  34
Entities with count  10–49:  270
Entities with count   1–9:   5442

Table raw_entity_counts written to wti_thesis.db  (last_updated: 2026-06-01T08:32:44Z)


## Cell 3 — Triage (query `raw_entity_counts`)

Review the distribution in three passes and update `03_src/nlp/llm_features.py` accordingly:

```sql
-- useful during triage
SELECT raw_entity, count FROM raw_entity_counts WHERE count >= 100 ORDER BY count DESC;
SELECT raw_entity, count FROM raw_entity_counts WHERE count BETWEEN 50 AND 99 ORDER BY count DESC;
```

- **≥100 occurrences**: map all variants of known canonical entities; promote new entities with clear supply/demand/risk signal to `CANONICAL_ENTITIES`.
- **50–99 occurrences**: apply the same rules; this is the minimum threshold for adding a new canonical.
- **Long tail (<50)**: document drop rationale. These are silently discarded by `canonicalize_entities()`.

Once `llm_features.py` is updated, continue to Cell 4.

## Cell 4 — Reload module and apply canonicalization

After updating `ENTITY_CANONICAL` and `CANONICAL_ENTITIES` in `03_src/nlp/llm_features.py`,
reload the module here so changes take effect without restarting the kernel.
Check coverage stats and the 10-row sample before writing to the DB.

In [24]:
import importlib
import nlp.llm_features as _llm_mod
importlib.reload(_llm_mod)
from nlp.llm_features import ENTITY_CANONICAL, CANONICAL_ENTITIES, canonicalize_entities

print(f'ENTITY_CANONICAL variants:  {len(ENTITY_CANONICAL):,}')
print(f'CANONICAL_ENTITIES:         {len(CANONICAL_ENTITIES):,}')

df['canonical'] = df['entities'].apply(canonicalize_entities)

n_with_canonical = (df['canonical'].apply(len) > 0).sum()
n_empty = len(df) - n_with_canonical
total_rows = df['canonical'].apply(len).sum()

print(f'\nArticles with >= 1 canonical entity: {n_with_canonical:,}  ({n_with_canonical/len(df)*100:.1f}%)')
print(f'Articles with 0 canonical matches:   {n_empty:,}  ({n_empty/len(df)*100:.1f}%)')
print(f'Total (article_id, canonical) pairs:  {total_rows:,}')
print()
print('Sample — raw entities vs canonical (10 rows):')
sample = df[df['canonical'].apply(len) > 0].iloc[30:40][['article_id', 'entities', 'canonical']]
for _, row in sample.iterrows():
    raw = json.loads(row['entities'])
    print(f'  {row["article_id"]:>6}  raw={raw}')
    print(f'         canonical={row["canonical"]}')

ENTITY_CANONICAL variants:  121
CANONICAL_ENTITIES:         70

Articles with >= 1 canonical entity: 10,268  (87.9%)
Articles with 0 canonical matches:   1,407  (12.1%)
Total (article_id, canonical) pairs:  29,357

Sample — raw entities vs canonical (10 rows):
      89  raw=['India', 'Russia', 'Hardeep Singh Puri']
         canonical=['India', 'Russia']
      90  raw=['OPEC+', 'Russia', 'Hamas', 'Israel', 'Houthi', 'Yemen', 'Iran', 'European Union', 'G7']
         canonical=['OPEC+', 'Russia', 'Hamas', 'Israel', 'Houthis', 'Yemen', 'Iran', 'EU']
      95  raw=['Russia', 'India', 'Africa', 'Iraq', 'Saudi Arabia', 'Vortexa']
         canonical=['Russia', 'India', 'Iraq', 'Saudi Arabia']
      98  raw=['Donald Trump', 'Iran', 'United States', 'Canada']
         canonical=['Trump', 'Iran', 'US', 'Canada']
     100  raw=['Biden administration', 'United States', 'Russia', 'Iran', 'Venezuela', 'Rachel Ziemba', 'Center for a New American Security', 'Fernando Ferreira', 'Rapidan Energy Group']


## Cell 5 — Write to article_entities

**Verify Cell 4 coverage and sample output look correct before running this cell.**

In [25]:
conn = sqlite3.connect(DB_PATH)

conn.execute("DROP TABLE IF EXISTS article_entities")
conn.execute("""
    CREATE TABLE article_entities (
        article_id       INTEGER NOT NULL,
        canonical_entity TEXT    NOT NULL,
        PRIMARY KEY (article_id, canonical_entity)
    )
""")
conn.commit()

rows_written = 0
for _, row in df.iterrows():
    for entity in row['canonical']:
        conn.execute(
            "INSERT OR IGNORE INTO article_entities (article_id, canonical_entity) VALUES (?, ?)",
            (int(row['article_id']), entity)
        )
        rows_written += 1

conn.commit()

count = conn.execute("SELECT COUNT(*) FROM article_entities").fetchone()[0]
n_articles = conn.execute("SELECT COUNT(DISTINCT article_id) FROM article_entities").fetchone()[0]
n_entities = conn.execute("SELECT COUNT(DISTINCT canonical_entity) FROM article_entities").fetchone()[0]
conn.close()

print(f'Rows written:              {rows_written:,}')
print(f'Rows in article_entities:  {count:,}')
print(f'Distinct articles:         {n_articles:,}')
print(f'Distinct canonical entities used: {n_entities:,}  (out of {len(CANONICAL_ENTITIES)} in list)')

Rows written:              29,357
Rows in article_entities:  29,357
Distinct articles:         10,268
Distinct canonical entities used: 70  (out of 70 in list)


## Cell 6 — Verification

Three checks:
1. Top 20 canonical entities by article count — sanity-check frequency ordering.
2. Five random sample articles: original `entities` JSON vs canonical tags.
3. Coverage rate: what share of total raw entity mentions got a canonical mapping.

In [26]:
conn = sqlite3.connect(DB_PATH)

print('=== Top 20 canonical entities by article count ===')
freq = pd.read_sql("""
    SELECT canonical_entity, COUNT(*) AS n_articles
    FROM article_entities
    GROUP BY canonical_entity
    ORDER BY n_articles DESC
    LIMIT 20
""", conn)
print(freq.to_string(index=False))

print()
print('=== 5 sample articles: raw entities → canonical tags ===')
sample = pd.read_sql("""
    SELECT lf.article_id,
           a.title,
           lf.entities AS raw_entities,
           GROUP_CONCAT(ae.canonical_entity, ' | ') AS canonical_tags
    FROM llm_features lf
    JOIN articles a ON a.id = lf.article_id
    JOIN article_entities ae ON ae.article_id = lf.article_id
    WHERE lf.usable = 1
    GROUP BY lf.article_id
    ORDER BY RANDOM()
    LIMIT 5
""", conn)

for _, row in sample.iterrows():
    print(f'  [{row["article_id"]}] {row["title"][:70]}')
    print(f'    raw:       {row["raw_entities"]}')
    print(f'    canonical: {row["canonical_tags"]}')
    print()

conn.close()

# Coverage: two well-defined metrics derived from raw_entity_counts
_canonical_set = set(CANONICAL_ENTITIES)
is_mapped = df_raw['raw_entity'].apply(
    lambda e: bool(ENTITY_CANONICAL.get(e) and ENTITY_CANONICAL[e] in _canonical_set)
)
n_strings_total  = len(df_raw)
n_strings_mapped = int(is_mapped.sum())
mentions_total   = int(df_raw['count'].sum())
mentions_mapped  = int(df_raw.loc[is_mapped, 'count'].sum())

print('=== Coverage ===')
print(f'Distinct raw strings mapped:  {n_strings_mapped:>5} / {n_strings_total}  ({n_strings_mapped/n_strings_total*100:.1f}%)')
print(f'Mention-level coverage:       {mentions_mapped:>6,} / {mentions_total:,}  ({mentions_mapped/mentions_total*100:.1f}%)')

=== Top 20 canonical entities by article count ===
canonical_entity  n_articles
              US        4487
            Iran        3201
           Trump        1935
          Russia        1743
           China        1584
           OPEC+        1382
          Israel        1099
           India         940
            OPEC         845
Strait of Hormuz         832
    Saudi Arabia         705
             Fed         687
         Ukraine         685
       Venezuela         631
             WTI         624
     Middle East         581
           Brent         574
             EIA         457
             UAE         424
             IEA         411

=== 5 sample articles: raw entities → canonical tags ===
  [2307] Oil price news : Oil fluctuates as traders eye OPEC+ cutbacks and Geop
    raw:       ["Donald Trump", "Iran", "U.S.", "Canada"]
    canonical: Canada | Iran | Trump | US

  [17911] US Crude Oil Inventory Build Pressures Prices
    raw:       ["American Petroleum Institute